# K-Nearest Neighbors Experiments

This notebook documents all KNN experiments on the fraud detection dataset.
Each experiment is run on **two feature sets** to compare the effect of feature selection:

| Label | Source | Features |
|-------|--------|----------|
| **Partially Selected (PS)** | `data/processed/partially_selected_features.csv` | 33 features (all after column cleanup) |
| **Selected (S)** | `data/processed/selected_features.csv` | 20 features (MRMR-selected subset) |

**Split strategy:** 70% train / 15% validation / 15% test (stratified, `random_state=42`)

Both pipelines use the same `random_state=42` and identical split ratios, so the same rows
land in train/validation/test. The only difference is the feature set.

**Why KNN?** KNN is a non-parametric, instance-based learner that makes no assumptions about
the underlying data distribution. Unlike logistic regression (linear boundary) or Random Forest
(tree-based splits), KNN classifies by finding the K closest training samples and taking a
majority vote. This makes it a useful baseline for understanding how well local similarity
in feature space predicts fraud.

**Note on scaling:** KNN is distance-based, so features on different scales would dominate the
distance calculation. StandardScaler is applied (fit on train only) to ensure all features
contribute equally.

**Note on class imbalance:** KNN has no built-in `class_weight` parameter. On imbalanced data,
the majority class (non-fraud) dominates neighborhoods, causing KNN to predict non-fraud at
the default 0.5 threshold. **Threshold tuning is critical** — we tune the classification
threshold jointly with K to find the best operating point.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_curve, auc,
    precision_recall_curve, f1_score
)
import warnings
warnings.filterwarnings('ignore')

from src.data.load_data import load_selected_features
from src.data.preprocess import encode_target, encode_categoricals, scale_features, split_data
from src.models.predict import predict, predict_proba
from src.evaluation.metrics import evaluate_model, print_metrics

print('Imports loaded successfully.')

## Data Preparation

We load both feature sets and apply the same preprocessing pipeline:
1. **Label encoding** for categorical features — converts string categories to integers
2. **Stratified train/val/test split** — 70/15/15 ratio, preserving class balance in each set
3. **Standard scaling** — fit on training set only, then transform val/test to prevent data leakage

**Why scale for KNN?** KNN uses distance (e.g. Euclidean) to find nearest neighbors. Without
scaling, features with large ranges (e.g. `total_claim_amount` in thousands) dominate the
distance calculation, making features with small ranges (e.g. `witnesses` in 0-5) irrelevant.
StandardScaler ensures each feature contributes proportionally to the distance.

In [ ]:
def prepare_dataset(path, label):
    df = load_selected_features(path)
    print(f"\n{'='*60}")
    print(f"{label} — {path}")
    print(f"{'='*60}")
    print(f"Shape: {df.shape}")
    print(f"Class distribution:\n{df['fraud_reported'].value_counts()}")
    print(f"Fraud rate: {(df['fraud_reported'] == 'Y').mean()*100:.1f}%")

    X = df.drop(columns=['fraud_reported'])
    y = encode_target(df['fraud_reported'])

    X_encoded, encoders = encode_categoricals(X)
    cat_cols = X.select_dtypes(include=['object']).columns.tolist()
    num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
    print(f"\nCategorical features ({len(cat_cols)}): {cat_cols}")
    print(f"Numerical features ({len(num_cols)}): {num_cols}")

    X_train, X_val, X_test, y_train, y_val, y_test = split_data(
        X_encoded, y, test_size=0.15, val_size=0.15, random_state=42
    )
    X_train_s, X_val_s, X_test_s, scaler = scale_features(X_train, X_val, X_test)
    print(f"\nTrain: {X_train.shape[0]}  |  Val: {X_val.shape[0]}  |  Test: {X_test.shape[0]}")
    print(f"Scaling: fit on train, transformed all sets")

    return {
        'X_train': X_train_s, 'X_val': X_val_s, 'X_test': X_test_s,
        'y_train': y_train, 'y_val': y_val, 'y_test': y_test,
        'scaler': scaler, 'label': label,
    }

ds_ps = prepare_dataset('../data/processed/partially_selected_features.csv', 'Partially Selected (33 feat.)')
ds_s  = prepare_dataset('../data/processed/selected_features.csv', 'Selected / MRMR (20 feat.)')

datasets = [ds_ps, ds_s]

In [ ]:
# Utility: find the threshold that maximizes F1 on a given set
def find_best_threshold(y_true, y_prob, thresholds=np.arange(0.05, 0.96, 0.05)):
    best_t, best_f1 = 0.5, 0
    for t in thresholds:
        yp = (y_prob >= t).astype(int)
        if yp.sum() == 0 or yp.sum() == len(yp):
            continue
        f = f1_score(y_true, yp)
        if f > best_f1:
            best_t, best_f1 = t, f
    return best_t, best_f1

---
## Run 1: Baseline KNN (K=5)

**Goal:** Establish a baseline with the most common default: K=5, uniform weights, Euclidean distance.

**Configuration:**
- `n_neighbors=5` — a common default. Each sample is classified by the majority vote of its 5
  nearest neighbors in the scaled feature space.
- `weights='uniform'` — all 5 neighbors contribute equally regardless of distance.
- `metric='euclidean'` — straight-line distance in the scaled feature space.

**Expected challenge:** KNN has no `class_weight` parameter. With ~75% non-fraud in the training
set, most neighborhoods will be majority non-fraud, so the model will predict non-fraud
too often at the default 0.5 threshold. This is a known limitation we address in later runs.

In [ ]:
for ds in datasets:
    model = KNeighborsClassifier(
        n_neighbors=5, weights='uniform', metric='euclidean', n_jobs=-1
    )
    model.fit(ds['X_train'], ds['y_train'])
    ds['model_v1'] = model
    ds['results_log'] = []

    print(f"\n{'='*60}")
    print(f"{ds['label']}")
    print(f"{'='*60}")
    for name, X_set, y_set in [('Train', ds['X_train'], ds['y_train']),
                                ('Validation', ds['X_val'], ds['y_val'])]:
        y_pred = predict(model, X_set)
        y_prob = predict_proba(model, X_set)
        res = evaluate_model(y_set, y_pred, y_prob)
        print_metrics(res, name)
        print()
        if name == 'Validation':
            ds['val_res_v1'] = res

    ds['results_log'].append({
        'Run': 'V1: Baseline (K=5, uniform, euclidean)',
        'Val F1': ds['val_res_v1']['f1'],
        'Val Precision': ds['val_res_v1']['precision'],
        'Val Recall': ds['val_res_v1']['recall'],
        'Val ROC AUC': ds['val_res_v1']['roc_auc'],
    })

In [ ]:
# Confusion matrices — baseline, side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, ds in zip(axes, datasets):
    y_pred = predict(ds['model_v1'], ds['X_val'])
    cm = confusion_matrix(ds['y_val'], y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Non-Fraud', 'Fraud'], yticklabels=['Non-Fraud', 'Fraud'])
    ax.set_title(f'V1 Baseline (K=5) — {ds["label"]}')
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')

plt.tight_layout()
plt.show()

---
## Run 2: Cross-Validation to Assess Stability

**Goal:** Assess how stable the baseline KNN is across different data folds.

**Method:** 5-fold stratified CV on the training set.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for ds in datasets:
    print(f"\n{'='*60}")
    print(f"{ds['label']}")
    print(f"{'='*60}")

    cv_model = KNeighborsClassifier(
        n_neighbors=5, weights='uniform', metric='euclidean', n_jobs=-1
    )
    for metric in ['f1', 'precision', 'recall', 'roc_auc']:
        scores = cross_val_score(cv_model, ds['X_train'], ds['y_train'], cv=cv, scoring=metric)
        print(f"{metric:<12}: {scores.mean():.4f} (+/- {scores.std():.4f})  | folds: {[f'{s:.3f}' for s in scores]}")

print("\nNote: Low F1/recall at default threshold is expected — KNN has no class weighting.")
print("ROC AUC gives a threshold-independent view of the model's ranking quality.")

---
## Run 3: K Tuning (n_neighbors) with Threshold Optimization

**Goal:** Find the optimal K value.

**Why joint K + threshold tuning?** At the default threshold of 0.5, KNN on imbalanced data
heavily favors the majority class. With K=5, a sample needs 3+ fraud neighbors to be classified
as fraud — unlikely when only 25% of training data is fraud. Higher K values produce smoother
probability estimates (K=51 means probabilities range from 0/51 to 51/51 in steps of 1/51),
which gives threshold tuning more room to find a good operating point.

**Method:** For each K, we find the F1-optimal threshold on the validation set. The best
K is the one whose optimal threshold achieves the highest F1.

In [ ]:
k_values = [1, 3, 5, 7, 9, 11, 15, 21, 31, 51]

for ds in datasets:
    k_results = []
    best_k, best_k_f1, best_k_t = 5, 0, 0.5

    for k in k_values:
        model_k = KNeighborsClassifier(
            n_neighbors=k, weights='uniform', metric='euclidean', n_jobs=-1
        )
        model_k.fit(ds['X_train'], ds['y_train'])

        y_pred_val = predict(model_k, ds['X_val'])
        y_prob_val = predict_proba(model_k, ds['X_val'])
        val_res = evaluate_model(ds['y_val'], y_pred_val, y_prob_val)

        y_pred_train = predict(model_k, ds['X_train'])
        train_f1 = f1_score(ds['y_train'], y_pred_train)

        # Find best threshold for this K
        bt, bf = find_best_threshold(ds['y_val'], y_prob_val)
        if bf > best_k_f1:
            best_k, best_k_f1, best_k_t = k, bf, bt

        k_results.append({
            'K': k,
            'Train F1': train_f1,
            'Val F1 (t=0.5)': val_res['f1'],
            'Best Threshold': bt,
            'Val F1 (tuned)': bf,
            'Val ROC AUC': val_res['roc_auc'],
        })

    k_df = pd.DataFrame(k_results)
    ds['k_df'] = k_df
    ds['best_k'] = best_k
    ds['best_k_t'] = best_k_t

    print(f"\n{'='*60}")
    print(f"{ds['label']}")
    print(f"{'='*60}")
    print(k_df.to_string(index=False))
    print(f"\nBest K={best_k} at threshold={best_k_t:.2f} (Val F1={best_k_f1:.4f})")

In [ ]:
# Visualize K tuning — both feature sets
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, ds in zip(axes, datasets):
    k_df = ds['k_df']
    ax.plot(k_df['K'], k_df['Train F1'], 'o-', label='Train F1')
    ax.plot(k_df['K'], k_df['Val F1 (t=0.5)'], 's-', label='Val F1 (t=0.5)')
    ax.plot(k_df['K'], k_df['Val F1 (tuned)'], '^-', label='Val F1 (tuned t)', linewidth=2)
    ax.plot(k_df['K'], k_df['Val ROC AUC'], 'D-', label='Val ROC AUC', alpha=0.6)
    ax.axvline(x=ds['best_k'], color='red', linestyle='--', alpha=0.5,
               label=f'Best K={ds["best_k"]}')
    ax.set_xlabel('K (n_neighbors)')
    ax.set_ylabel('Score')
    ax.set_title(f'K Tuning — {ds["label"]}')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Key insight: Val F1 at default threshold (t=0.5) is misleadingly low.")
print("With threshold tuning, higher K values actually perform much better.")
print("ROC AUC (threshold-independent) confirms that higher K improves ranking quality.")

---
## Run 4: Weighting Scheme (uniform vs distance)

**Goal:** Test whether distance-weighted voting improves over uniform voting.

**Why distance weighting?** With `weights='distance'`, closer neighbors have more influence
on the vote than farther ones. This can help when the K nearest neighbors span a range of
distances — the closest few (most similar) should matter more.

**Method:** Fix K at the best value from Run 3 and compare uniform vs distance weighting.

In [ ]:
for ds in datasets:
    best_w, best_w_f1, best_w_t = 'uniform', 0, 0.5

    print(f"\n{'='*60}")
    print(f"{ds['label']} (K={ds['best_k']})")
    print(f"{'='*60}")

    for w in ['uniform', 'distance']:
        m = KNeighborsClassifier(
            n_neighbors=ds['best_k'], weights=w, metric='euclidean', n_jobs=-1
        )
        m.fit(ds['X_train'], ds['y_train'])
        r = evaluate_model(ds['y_val'], predict(m, ds['X_val']), predict_proba(m, ds['X_val']))
        y_prob_val = predict_proba(m, ds['X_val'])
        bt, bf = find_best_threshold(ds['y_val'], y_prob_val)
        if bf > best_w_f1:
            best_w, best_w_f1, best_w_t = w, bf, bt
        print(f"  {w:<10}: Val F1(t=0.5)={r['f1']:.4f}  Val F1(t={bt:.2f})={bf:.4f}  AUC={r['roc_auc']:.4f}")

    ds['best_w'] = best_w
    print(f"  Best weights: {best_w}")

---
## Run 5: Distance Metric (Euclidean vs Manhattan)

**Goal:** Test whether Manhattan distance outperforms Euclidean.

**Why Manhattan?** Euclidean distance is sensitive to outliers (squared differences amplify
large deviations). Manhattan distance (sum of absolute differences) is more robust.
Additionally, in high-dimensional spaces, Manhattan distance tends to preserve relative
differences better than Euclidean (a phenomenon related to the curse of dimensionality).

**Method:** Fix K and weights at their best values, compare distance metrics.

In [ ]:
for ds in datasets:
    best_met, best_met_f1, best_met_t = 'euclidean', 0, 0.5

    print(f"\n{'='*60}")
    print(f"{ds['label']} (K={ds['best_k']}, weights={ds['best_w']})")
    print(f"{'='*60}")

    for met in ['euclidean', 'manhattan']:
        m = KNeighborsClassifier(
            n_neighbors=ds['best_k'], weights=ds['best_w'], metric=met, n_jobs=-1
        )
        m.fit(ds['X_train'], ds['y_train'])
        r = evaluate_model(ds['y_val'], predict(m, ds['X_val']), predict_proba(m, ds['X_val']))
        y_prob_val = predict_proba(m, ds['X_val'])
        bt, bf = find_best_threshold(ds['y_val'], y_prob_val)
        if bf > best_met_f1:
            best_met, best_met_f1, best_met_t = met, bf, bt
        print(f"  {met:<12}: Val F1(t=0.5)={r['f1']:.4f}  Val F1(t={bt:.2f})={bf:.4f}  AUC={r['roc_auc']:.4f}")

    ds['best_met'] = best_met
    print(f"  Best metric: {best_met}")

---
## Run 6: Best Configuration with CV Confirmation

**Goal:** Confirm the best hyperparameters found in Runs 3-5 via cross-validation.

**Method:** 5-fold stratified CV with the best K, weights, and distance metric.

In [ ]:
for ds in datasets:
    model_v2 = KNeighborsClassifier(
        n_neighbors=ds['best_k'], weights=ds['best_w'], metric=ds['best_met'], n_jobs=-1
    )

    cv_f1 = cross_val_score(model_v2, ds['X_train'], ds['y_train'], cv=cv, scoring='f1')
    cv_auc = cross_val_score(model_v2, ds['X_train'], ds['y_train'], cv=cv, scoring='roc_auc')

    print(f"\n{'='*60}")
    print(f"{ds['label']}")
    print(f"{'='*60}")
    print(f"Config: K={ds['best_k']}, weights={ds['best_w']}, metric={ds['best_met']}")
    print(f"CV F1 (default threshold): {cv_f1.mean():.4f} (+/- {cv_f1.std():.4f})")
    print(f"CV ROC AUC: {cv_auc.mean():.4f} (+/- {cv_auc.std():.4f})")

    model_v2.fit(ds['X_train'], ds['y_train'])
    ds['model_v2'] = model_v2

    for name, X_set, y_set in [('Train', ds['X_train'], ds['y_train']),
                                ('Validation', ds['X_val'], ds['y_val'])]:
        y_pred = predict(model_v2, X_set)
        y_prob = predict_proba(model_v2, X_set)
        res = evaluate_model(y_set, y_pred, y_prob)
        print_metrics(res, name)
        print()
        if name == 'Validation':
            ds['val_res_v2'] = res

    ds['results_log'].append({
        'Run': f'V2: Tuned (K={ds["best_k"]}, {ds["best_w"]}, {ds["best_met"]})',
        'Val F1': ds['val_res_v2']['f1'],
        'Val Precision': ds['val_res_v2']['precision'],
        'Val Recall': ds['val_res_v2']['recall'],
        'Val ROC AUC': ds['val_res_v2']['roc_auc'],
    })

---
## Run 7: Threshold Tuning

**Goal:** Find the F1-optimal classification threshold.

**Why this is especially important for KNN:** Unlike Random Forest or logistic regression where
`class_weight='balanced'` adjusts for imbalance internally, KNN has no such mechanism. The only
way to handle the majority-class bias is to lower the classification threshold — accept a sample
as fraud even if less than 50% of its neighbors are fraud.

**Method:** Sweep thresholds from 0.05 to 0.95 on the validation set.

In [ ]:
thresholds = np.arange(0.05, 0.96, 0.05)

for ds in datasets:
    y_prob_val = predict_proba(ds['model_v2'], ds['X_val'])
    threshold_results = []

    for t in thresholds:
        y_pred_t = (y_prob_val >= t).astype(int)
        if y_pred_t.sum() == 0 or y_pred_t.sum() == len(y_pred_t):
            continue
        res = evaluate_model(ds['y_val'], y_pred_t, y_prob_val)
        threshold_results.append({
            'Threshold': t,
            'Precision': res['precision'],
            'Recall': res['recall'],
            'F1': res['f1'],
            'Accuracy': res['accuracy'],
        })

    t_df = pd.DataFrame(threshold_results)
    ds['t_df'] = t_df
    ds['best_threshold'] = t_df.loc[t_df['F1'].idxmax(), 'Threshold']

    print(f"\n{'='*60}")
    print(f"{ds['label']}")
    print(f"{'='*60}")
    print(t_df.to_string(index=False))
    print(f"\nBest threshold by F1: {ds['best_threshold']:.2f}")

In [ ]:
# Visualize threshold trade-off — both feature sets
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, ds in zip(axes, datasets):
    t_df = ds['t_df']
    ax.plot(t_df['Threshold'], t_df['Precision'], 'o-', label='Precision')
    ax.plot(t_df['Threshold'], t_df['Recall'], 's-', label='Recall')
    ax.plot(t_df['Threshold'], t_df['F1'], '^-', label='F1', linewidth=2)
    ax.axvline(x=ds['best_threshold'], color='red', linestyle='--', alpha=0.5,
               label=f'Best={ds["best_threshold"]:.2f}')
    ax.axvline(x=0.5, color='gray', linestyle=':', alpha=0.5, label='Default (0.5)')
    ax.set_xlabel('Classification Threshold')
    ax.set_ylabel('Score')
    ax.set_title(f'Threshold Tuning — {ds["label"]}')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Evaluate with best threshold on train + validation (test held out)
for ds in datasets:
    best_threshold = ds['best_threshold']
    print(f"\n{'='*60}")
    print(f"{ds['label']}  —  threshold = {best_threshold:.2f}")
    print(f"{'='*60}")

    for name, X_set, y_set in [('Train', ds['X_train'], ds['y_train']),
                                ('Validation', ds['X_val'], ds['y_val'])]:
        y_prob = predict_proba(ds['model_v2'], X_set)
        y_pred_t = (y_prob >= best_threshold).astype(int)
        res = evaluate_model(y_set, y_pred_t, y_prob)
        print_metrics(res, name)
        print()
        if name == 'Validation':
            ds['val_res_v3'] = res

    ds['results_log'].append({
        'Run': f'V3: Tuned + threshold={best_threshold:.2f}',
        'Val F1': ds['val_res_v3']['f1'],
        'Val Precision': ds['val_res_v3']['precision'],
        'Val Recall': ds['val_res_v3']['recall'],
        'Val ROC AUC': ds['val_res_v3']['roc_auc'],
    })

---
## Run Summary & Model Selection

Compare all experiment runs on **validation metrics only** for each feature set.

In [ ]:
for ds in datasets:
    summary_df = pd.DataFrame(ds['results_log'])
    ds['summary_df'] = summary_df

    print(f"\n{'='*90}")
    print(f"EXPERIMENT LOG — {ds['label']}")
    print(f"{'='*90}")
    print(summary_df.to_string(index=False))

    best_idx = summary_df['Val F1'].idxmax()
    ds['best_run_name'] = summary_df.loc[best_idx, 'Run']
    print(f"\nBest run by Val F1: {ds['best_run_name']}")
    print(f"  Val F1:        {summary_df.loc[best_idx, 'Val F1']:.4f}")
    print(f"  Val Precision: {summary_df.loc[best_idx, 'Val Precision']:.4f}")
    print(f"  Val Recall:    {summary_df.loc[best_idx, 'Val Recall']:.4f}")
    print(f"  Val ROC AUC:   {summary_df.loc[best_idx, 'Val ROC AUC']:.4f}")

---
## Cross-Dataset Comparison (Validation Only)

Side-by-side comparison of the best run from each feature set.

In [ ]:
comparison_rows = []
for ds in datasets:
    best_idx = ds['summary_df']['Val F1'].idxmax()
    row = ds['summary_df'].iloc[best_idx].to_dict()
    row['Feature Set'] = ds['label']
    comparison_rows.append(row)

comp_df = pd.DataFrame(comparison_rows)
print("="*90)
print("BEST MODEL COMPARISON — Partially Selected (33) vs Selected/MRMR (20)")
print("="*90)
print(comp_df[['Feature Set', 'Run', 'Val F1', 'Val Precision', 'Val Recall', 'Val ROC AUC']].to_string(index=False))

for i in range(len(comp_df)):
    for j in range(i+1, len(comp_df)):
        r1, r2 = comp_df.iloc[i], comp_df.iloc[j]
        print(f"\nDelta ({r1['Feature Set'][:15]} vs {r2['Feature Set'][:15]}):")
        for m in ['Val F1', 'Val Precision', 'Val Recall', 'Val ROC AUC']:
            delta = r1[m] - r2[m]
            sign = '+' if delta >= 0 else ''
            print(f"  {m}: {sign}{delta:.4f}")

In [ ]:
# Visual comparison — bar chart of validation metrics
metrics = ['Val F1', 'Val Precision', 'Val Recall', 'Val ROC AUC']
metric_labels = ['F1', 'Precision', 'Recall', 'ROC AUC']
bar_colors = ['#3498db', '#e74c3c', '#2ecc71']

x = np.arange(len(metrics))
n_bars = len(comp_df)
width = 0.8 / n_bars

fig, ax = plt.subplots(figsize=(12, 5))
for idx, (_, row) in enumerate(comp_df.iterrows()):
    vals = [row[m] for m in metrics]
    bars = ax.bar(x + (idx - n_bars/2 + 0.5) * width, vals, width,
                  label=row['Feature Set'][:20], color=bar_colors[idx % len(bar_colors)])
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.3f}', xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3), textcoords='offset points', ha='center', fontsize=8)

ax.set_ylabel('Score')
ax.set_title('Best Model Comparison — Validation Set')
ax.set_xticks(x)
ax.set_xticklabels(metric_labels)
ax.legend(fontsize=8)
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

---
## Final Test Evaluation (ONE-TIME)

**This is the only place in the notebook where the test set is used.**

After all hyperparameter tuning and model selection on the validation set, we evaluate the
best configuration on the held-out test set exactly once for each feature set.

In [ ]:
for ds in datasets:
    best_threshold = ds['best_threshold']

    print(f"\n{'='*60}")
    print(f"FINAL TEST — {ds['label']}")
    print(f"{'='*60}")
    print(f"Model: KNN(K={ds['best_k']}, weights={ds['best_w']}, metric={ds['best_met']})")
    print(f"Threshold: {best_threshold:.2f}\n")

    y_prob_test = predict_proba(ds['model_v2'], ds['X_test'])
    y_pred_test = (y_prob_test >= best_threshold).astype(int)
    test_res = evaluate_model(ds['y_test'], y_pred_test, y_prob_test)
    print_metrics(test_res, 'Test')

    print("\nClassification Report:")
    print(classification_report(ds['y_test'], y_pred_test, target_names=['Non-Fraud', 'Fraud']))

    ds['test_res'] = test_res
    ds['y_prob_test'] = y_prob_test
    ds['y_pred_test'] = y_pred_test

In [ ]:
# Test set comparison table
print("="*80)
print("FINAL TEST COMPARISON — Partially Selected vs Selected/MRMR")
print("="*80)

test_comp = []
for ds in datasets:
    test_comp.append({
        'Feature Set': ds['label'],
        'Test F1': ds['test_res']['f1'],
        'Test Precision': ds['test_res']['precision'],
        'Test Recall': ds['test_res']['recall'],
        'Test ROC AUC': ds['test_res']['roc_auc'],
        'Test Accuracy': ds['test_res']['accuracy'],
    })

test_comp_df = pd.DataFrame(test_comp)
print(test_comp_df.to_string(index=False))

for i in range(len(test_comp_df)):
    for j in range(i+1, len(test_comp_df)):
        r1, r2 = test_comp_df.iloc[i], test_comp_df.iloc[j]
        print(f"\nDelta ({r1['Feature Set'][:15]} vs {r2['Feature Set'][:15]}):")        
        for m in ['Test F1', 'Test Precision', 'Test Recall', 'Test ROC AUC', 'Test Accuracy']:
            delta = r1[m] - r2[m]
            sign = '+' if delta >= 0 else ''
            print(f"  {m}: {sign}{delta:.4f}")

In [ ]:
# Final visualizations — ROC, PR curve, confusion matrices
n_ds = len(datasets)
fig, axes = plt.subplots(2, max(n_ds, 2), figsize=(7*max(n_ds, 2), 10))
colors = ['#3498db', '#e74c3c', '#2ecc71']

# Row 0: ROC + PR
for i, ds in enumerate(datasets):
    fpr, tpr, _ = roc_curve(ds['y_test'], ds['y_prob_test'])
    roc_auc_val = auc(fpr, tpr)
    axes[0, 0].plot(fpr, tpr, color=colors[i], linewidth=2,
                    label=f'{ds["label"][:15]} (AUC={roc_auc_val:.3f})')
axes[0, 0].plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random')
axes[0, 0].set_xlabel('False Positive Rate'); axes[0, 0].set_ylabel('True Positive Rate')
axes[0, 0].set_title('ROC Curves (Test Set)'); axes[0, 0].legend(fontsize=7); axes[0, 0].grid(True, alpha=0.3)

for i, ds in enumerate(datasets):
    prec, rec, _ = precision_recall_curve(ds['y_test'], ds['y_prob_test'])
    axes[0, 1].plot(rec, prec, color=colors[i], linewidth=2, label=ds['label'][:15])
axes[0, 1].axhline(y=datasets[0]['y_test'].mean(), color='gray', linestyle='--', alpha=0.5, label='Baseline')
axes[0, 1].set_xlabel('Recall'); axes[0, 1].set_ylabel('Precision')
axes[0, 1].set_title('PR Curves (Test Set)'); axes[0, 1].legend(fontsize=7); axes[0, 1].grid(True, alpha=0.3)

for j in range(2, max(n_ds, 2)):
    axes[0, j].set_visible(False)

# Row 1: Confusion matrices
for i, ds in enumerate(datasets):
    cm = confusion_matrix(ds['y_test'], ds['y_pred_test'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1, i],
                xticklabels=['Non-Fraud', 'Fraud'], yticklabels=['Non-Fraud', 'Fraud'])
    axes[1, i].set_title(f'CM — {ds["label"][:20]}')
    axes[1, i].set_ylabel('Actual'); axes[1, i].set_xlabel('Predicted')

plt.tight_layout()
plt.show()